# Pattern 4: Multi-Agent Supervisor Routing

This notebook demonstrates a supervisor that routes work between specialist agents.

In [13]:
from typing import TypedDict, Literal
import json
from pathlib import Path
from datetime import datetime, timezone

from langchain_ollama import ChatOllama
from langgraph.graph import END, START, StateGraph
from opentelemetry import trace
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor, ConsoleSpanExporter, SpanExportResult, SpanExporter

MODEL_NAME = "qwen3.5:2b"
llm = ChatOllama(model=MODEL_NAME, temperature=0, think=False, streaming=False)

RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
LOG_FILE = Path("trace_logs") / f"04_multi_agent_supervisor_spans_{RUN_TIMESTAMP}.log"


class FileSpanExporter(SpanExporter):
    def __init__(self, log_file: Path):
        self.log_file = log_file
        self.log_file.parent.mkdir(parents=True, exist_ok=True)

    def export(self, spans):
        with self.log_file.open("a", encoding="utf-8") as handle:
            for span in spans:
                record = {
                    "name": span.name,
                    "trace_id": f"{span.context.trace_id:032x}",
                    "span_id": f"{span.context.span_id:016x}",
                    "parent_span_id": f"{span.parent.span_id:016x}" if span.parent else None,
                    "start_time": datetime.fromtimestamp(span.start_time / 1_000_000_000, tz=timezone.utc).isoformat(),
                    "end_time": datetime.fromtimestamp(span.end_time / 1_000_000_000, tz=timezone.utc).isoformat() if span.end_time else None,
                    "attributes": dict(span.attributes),
                    "status": str(span.status.status_code.name),
                }
                handle.write(json.dumps(record, ensure_ascii=False) + "\n")
        return SpanExportResult.SUCCESS

    def shutdown(self):
        return None

    def force_flush(self, timeout_millis: int = 30000):
        return True


resource = Resource.create({"service.name": "agentic-patterns-notebook-04"})
current_provider = trace.get_tracer_provider()
if hasattr(current_provider, "add_span_processor"):
    provider = current_provider
    if not getattr(provider, "_agentic_patterns_notebook_04_configured", False):
        provider.add_span_processor(BatchSpanProcessor(ConsoleSpanExporter()))
        provider.add_span_processor(BatchSpanProcessor(FileSpanExporter(LOG_FILE)))
        setattr(provider, "_agentic_patterns_notebook_04_configured", True)
else:
    provider = TracerProvider(resource=resource)
    provider.add_span_processor(BatchSpanProcessor(ConsoleSpanExporter()))
    provider.add_span_processor(BatchSpanProcessor(FileSpanExporter(LOG_FILE)))
    setattr(provider, "_agentic_patterns_notebook_04_configured", True)
    trace.set_tracer_provider(provider)

tracer = trace.get_tracer("pattern-04.multi-agent-supervisor")


def invoke_llm_with_span(prompt: str, node_name: str, pattern_name: str = "multi_agent_supervisor") -> str:
    with tracer.start_as_current_span("llm.invoke") as span:
        span.set_attribute("agent.pattern", pattern_name)
        span.set_attribute("agent.node", node_name)
        span.set_attribute("llm.model", MODEL_NAME)
        span.set_attribute("llm.prompt_length", len(prompt))
        span.set_attribute("llm.prompt", prompt)
        response = llm.invoke(prompt)
        content = str(response.content)
        span.set_attribute("llm.response_length", len(content))
        span.set_attribute("llm.response", content)
        return content

In [14]:
class MultiAgentState(TypedDict):
    question: str
    research_notes: str
    draft_answer: str
    final_answer: str
    route: Literal['researcher', 'writer', 'finish']


def supervisor_node(state: MultiAgentState) -> MultiAgentState:
    route_prompt = f"""
You are a supervisor. Choose exactly one next route: researcher, writer, or finish.
Question: {state['question']}
Current research notes length: {len(state['research_notes'])}
Current draft length: {len(state['draft_answer'])}

Rules:
- If research notes are weak/empty, choose researcher.
- If research exists but no draft, choose writer.
- If draft exists and is enough, choose finish.
Return only one word.
"""
    with tracer.start_as_current_span("pattern.supervisor.routing") as pattern_span:
        pattern_span.set_attribute("agent.pattern", "multi_agent_supervisor")
        choice = invoke_llm_with_span(route_prompt, "supervisor").strip().lower()
    if 'researcher' in choice:
        state['route'] = 'researcher'
    elif 'writer' in choice:
        state['route'] = 'writer'
    else:
        state['route'] = 'finish'
    return state


def researcher_node(state: MultiAgentState) -> MultiAgentState:
    prompt = f"""
Collect concise research notes (5 bullets max) for this question:
{state['question']}
"""
    with tracer.start_as_current_span("pattern.researcher.step") as pattern_span:
        pattern_span.set_attribute("agent.pattern", "multi_agent_supervisor")
        state['research_notes'] = invoke_llm_with_span(prompt, "researcher")
    return state


def writer_node(state: MultiAgentState) -> MultiAgentState:
    prompt = f"""
Use these notes to draft a clear answer.
Question: {state['question']}
Notes:
{state['research_notes']}
"""
    with tracer.start_as_current_span("pattern.writer.step") as pattern_span:
        pattern_span.set_attribute("agent.pattern", "multi_agent_supervisor")
        state['draft_answer'] = invoke_llm_with_span(prompt, "writer")
    return state


def finalize_node(state: MultiAgentState) -> MultiAgentState:
    if state['draft_answer']:
        state['final_answer'] = state['draft_answer']
    else:
        state['final_answer'] = state['research_notes']
    return state


def route_from_supervisor(state: MultiAgentState) -> str:
    return state['route']

In [ ]:
builder = StateGraph(MultiAgentState)
builder.add_node('supervisor', supervisor_node)
builder.add_node('researcher', researcher_node)
builder.add_node('writer', writer_node)
builder.add_node('finalize', finalize_node)

builder.add_edge(START, 'supervisor')
builder.add_conditional_edges('supervisor', route_from_supervisor, {
    'researcher': 'researcher',
    'writer': 'writer',
    'finish': 'finalize',
})
builder.add_edge('researcher', 'supervisor')
builder.add_edge('writer', 'supervisor')
builder.add_edge('finalize', END)

graph = builder.compile()

initial = {
    'question': 'What are practical first projects to learn agentic AI on a laptop?',
    'research_notes': '',
    'draft_answer': '',
    'final_answer': '',
    'route': 'researcher',
}

result = graph.invoke(initial)
result['final_answer']

## Notes
This pattern scales to many specialists (tools, retrieval, coding, validation) with a stronger supervisor policy.